# Macro Voting Timing Model — Results Walkthrough
Spec: `report/BUILD_SPEC_v3.md` (frozen). This notebook is the narrative walkthrough
required by the deliverables list — it downloads the seven raw series, runs the
full pipeline in `src/`, and displays every required table and figure inline.

**Run this in Google Colab** (or any environment with real internet access to
`fred.stlouisfed.org` and `finance.yahoo.com`). It will NOT run in a network-locked
sandbox — `pandas_datareader` and `yfinance` need live egress to those two hosts.

In [ ]:
# --- Setup (Colab) ---
# If running from a zip/upload rather than a git clone, unzip into /content/macro-timing
# and uncomment the next two lines:
# import zipfile
# zipfile.ZipFile('macro-timing.zip').extractall('.')

!pip install -q pandas_datareader yfinance

import sys, os
sys.path.insert(0, os.path.abspath('.'))  # so `import src...` resolves

## 1. Run the full pipeline
Downloads + caches all 7 series to `data/raw/`, runs signals -> backtest -> evaluate, writes `output/metrics.json` and `output/figures/*.png`.

In [ ]:
from src.pipeline import run_all

results = run_all(force_refresh=False)  # set True to bypass the CSV cache and re-download
results["metrics_table"]

## 2. Tripwire check (gate G8)
Must read `tripped: False` before any result below is presented. If it's `True`, stop and complete a look-ahead audit — do not proceed to interpretation.

In [ ]:
results['tripwire']

## 3. Confusion matrix (Section 5.2)

In [ ]:
results['confusion_matrix']

## 4. Random-timing null (Section 5.3)
1,000 sims, fixed seed. Percentile of the real strategy within the null distribution:

In [ ]:
results['null_percentiles']

## 5. Robustness suite (Section 5.4)

In [ ]:
print("Threshold sensitivity (+/-0.3 / +/-0.5 / +/-0.7):")
display(results['threshold_table'])

print("\nSub-period metrics:")
display(results['subperiod_table'])

print("\nLeave-one-out factor ablation:")
display(results['ablation_table'])

## 6. Required figures

In [ ]:
from IPython.display import Image, display
import glob

for f in sorted(glob.glob('output/figures/*.png')):
    print(f)
    display(Image(filename=f))

## 7. Notes for REPORT.md

Once this cell block above runs clean with `tripwire.tripped == False`, copy the
printed numbers into `report/REPORT.md` Phase 5 (Performance) and Phase 6
(Interpretation). `report/REPORT.md` already contains the full Phase 1-4 economic
narrative and skeptic's memo — only Phase 5/6 need the real numbers dropped in.